# ЛР № 3. Профилирование и качество Seoul Bike Sharing Demand

Демонстрация участка `raw → staging`. Одна строка описывает один час работы системы велопроката.
Вопрос из [Project Brief](../docs/project_brief.md): как различается спрос на аренду велосипедов
по сезонам и часам суток?
Бизнес-ключ `(Date, Hour)` уникален в этом снимке; проверяем его уникальность и полностью совпадающие строки.

Запустите лабораторный стенд. Выберите ядро из `.venv` проекта.
Выполняйте ячейки сверху вниз. Для проверки повторяемости перезапустите ядро и выполните все ячейки ещё раз.

<a id="input"></a>
## Шаг 1. Входные данные и подключение

`ingested_on` - логическая дата загрузки, а не период наблюдений.
Аналитический период: `2017-12-01` – `2018-11-30` (один полный год).
Справочник зон в этих базовых проверках не используется.

Следующая ячейка определяет корень проекта при запуске из корня или каталога `notebooks`,
загружает настройки из `.env` без вывода секретов и использует подключение DuckDB из ЛР № 2.

In [1]:
import os
import sys
from html import escape
from pathlib import Path

import boto3
import duckdb
from dotenv import load_dotenv
from IPython.display import HTML, display

PROJECT_ROOT = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "pyproject.toml").is_file()
    and (parent / "src/analysis/read_raw.py").is_file()
)
load_dotenv(PROJECT_ROOT / ".env", override=True)
required = ("S3_ENDPOINT", "S3_ACCESS_KEY", "S3_SECRET_KEY")
missing = [name for name in required if not os.environ.get(name)]
if missing:
    raise ValueError("Не заданы настройки: " + ", ".join(missing))

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from analysis.read_raw import configure_s3  # noqa: E402

raw_path = "s3://raw/seoul_bike/ingested_on=2026-10-15/SeoulBikeData.csv"
period_start = "2017-12-01"
period_end = "2018-11-30"
source_date = raw_path.split("/ingested_on=", 1)[1].split("/", 1)[0]
output_prefix = f"s3://staging/seoul_bike/source_ingested_on={source_date}"

def literal(value):
    return "'" + value.replace("'", "''") + "'"

def show_table(headers, rows):
    header = "".join(f"<th>{escape(str(item))}</th>" for item in headers)
    body = "".join(
        "<tr>" + "".join(f"<td>{escape(str(item))}</td>" for item in row) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))

connection = duckdb.connect()
configure_s3(connection)
connection.execute(
    f"CREATE OR REPLACE TEMP VIEW raw_data AS SELECT * FROM read_csv_auto({literal(raw_path)})"
)
show_table(["Вход", "Начало периода", "Конец периода"],
           [(raw_path, period_start, period_end)])
show_table(["Date", "Hour", "Seasons", "Rented Bike Count"], connection.execute("""
    SELECT "Date", "Hour", "Seasons", "Rented Bike Count"
    FROM raw_data LIMIT 3
""").fetchall())

Вход,Начало периода,Конец периода
s3://raw/seoul_bike/ingested_on=2026-10-15/SeoulBikeData.csv,2017-12-01,2018-11-30


Date,Hour,Seasons,Rented Bike Count
2017-12-01,0,Winter,254
2017-12-01,1,Winter,204
2017-12-01,2,Winter,173


<a id="profile"></a>
## Шаг 2. Профиль исходных данных

Выбраны пять полей, необходимых для метрик из Project Brief:

- `Date` и `Hour` — временные срезы (сезон × час) и бизнес-ключ `(Date, Hour)`.
- `Seasons` — группировка спроса по сезонам.
- `Rented Bike Count` — основная метрика спроса (среднее, сумма, доля).
- `Functioning Day` — фильтр нерабочих часов, в которые спрос не наблюдался.

Профиль считается на сыром датасете до любых преобразований и исключений.
В исходном CSV 14 полей. При чтении S3-пути DuckDB автоматически получает дополнительное поле
`ingested_on` из имени каталога, поэтому при чтении видно 15 полей.

In [2]:
raw_count = connection.execute("SELECT count(*) FROM raw_data").fetchone()[0]
schema = {row[0]: row[1] for row in connection.execute("DESCRIBE raw_data").fetchall()}
show_table(["Строк", "Полей при чтении S3"], [(raw_count, len(schema))])

selected_fields = (
    "Date", "Hour", "Seasons", "Rented Bike Count", "Functioning Day",
)
profile_rows = []
for field in selected_fields:
    missing_count = connection.execute(
        f'SELECT count(*) FROM raw_data WHERE "{field}" IS NULL'
    ).fetchone()[0]
    examples = connection.execute(
        f'SELECT DISTINCT "{field}" FROM raw_data WHERE "{field}" IS NOT NULL ORDER BY 1 LIMIT 2'
    ).fetchall()
    share = f"{missing_count / raw_count:.3%}" if raw_count else "N/A"
    profile_rows.append((
        field, schema[field], ", ".join(str(row[0]) for row in examples),
        raw_count, missing_count, share,
    ))
show_table(["Поле", "Тип", "Примеры", "Проверено", "Пропусков", "Доля"], profile_rows)

Строк,Полей при чтении S3
8760,15


Поле,Тип,Примеры,Проверено,Пропусков,Доля
Date,DATE,"2017-12-01, 2017-12-02",8760,0,0.000%
Hour,BIGINT,"0, 1",8760,0,0.000%
Seasons,VARCHAR,"Autumn, Spring",8760,0,0.000%
Rented Bike Count,BIGINT,"0, 2",8760,0,0.000%
Functioning Day,BOOLEAN,"False, True",8760,0,0.000%


In [3]:
period = connection.execute(
    'SELECT min("Date"), max("Date") FROM raw_data'
).fetchone()
exact_duplicates = connection.execute("""
    SELECT count(*) - (SELECT count(*) FROM (SELECT DISTINCT * FROM raw_data))
    FROM raw_data
""").fetchone()[0]
show_table(["Минимальная дата", "Максимальная дата", "Полных повторов"],
           [(*period, exact_duplicates)])

key_duplicates = connection.execute("""
    SELECT COALESCE(sum(cnt - 1), 0) FROM (
        SELECT count(*) AS cnt FROM raw_data GROUP BY "Date", "Hour" HAVING count(*) > 1
    )
""").fetchone()[0]
show_table(["Повторов по бизнес-ключу (Date, Hour)"], [(key_duplicates,)])

seasons = connection.execute("""
    SELECT "Seasons", count(*) FROM raw_data GROUP BY 1 ORDER BY 1
""").fetchall()
show_table(["Сезон", "Записей"], seasons)

Минимальная дата,Максимальная дата,Полных повторов
2017-12-01,2018-11-30,0


"Повторов по бизнес-ключу (Date, Hour)"
0


Сезон,Записей
Autumn,2184
Spring,2208
Summer,2208
Winter,2160


### Наблюдения для отчёта

**Смысл пропуска по выбранным полям.**
Пропуск в `Date` или `Hour` означал бы невозможность отнести запись к конкретному часу,
построить сезонный или суточный срез и проверить уникальность бизнес-ключа.
Пропуск в `Rented Bike Count` — потерю основной метрики спроса.
Пропуск в `Seasons` — невозможность сезонного сравнения.
Пропуск в `Functioning Day` — невозможность отделить нерабочие часы от реального спроса.
В этом снимке пропусков нет ни в одном из выбранных полей (0 / 8 760 = 0,000 %).

**Повторы.**
Полные повторы считаются как `count(*)` минус `count(DISTINCT *)` по всем исходным полям.
Повторы по бизнес-ключу `(Date, Hour)` считаются как сумма `count(*) - 1` по группам с `count(*) > 1`.
Оба числа равны 0: 8 760 строк = 365 × 24, ключ `(Date, Hour)` уникален.

**Временное покрытие.**
Даты идут сплошным рядом от `2017-12-01` до `2018-11-30` без разрывов.
Это важно, потому что сезонные и суточные средние считаются по одинаково длинным рядам.

**Одно значимое распределение.**
Каждый час представлен ровно 365 записями, каждый сезон — 2 160–2 208 записями.
Значит, сравнение «сезон × час» в Project Brief не потребует дополнительной нормализации.

**Согласованность `Functioning Day`.**
Поле при чтении распознаётся как `BOOLEAN` со значениями `False` и `True`.
Логически `False` соответствует исходному `No`, `True` — `Yes`.
В нерабочие часы (`Functioning Day = False`) число аренд равно нулю.
Такие нули отражают недоступность сервиса, а не отсутствие спроса.
Формальная проверка согласованности выполняется в шаге 3.

<a id="rules"></a>
## Шаг 3. Правила качества

Шесть правил покрывают все требуемые свойства: уникальность (по бизнес-ключу), полноту значимого поля,
допустимость значений (диапазон часа, справочник сезонов, неотрицательность метрики)
и согласованность связанных полей (`Functioning Day` и `Rented Bike Count`).

Для каждого правила ниже указаны идентификатор, поля, проверяемые строки, условие, критичность,
действие и связь с задачей. Порог задан переменной в ячейке и участвует в сравнении.
Единица порога — число нарушений, допустимый максимум — ноль.

Порог оценивает набор. Действие по каждой нарушившей строке применяется независимо от превышения порога.
Доля считается внутри проверяемой совокупности; при пустой совокупности доля и сравнение — `N/A`.

In [4]:
rule_results = {}

def show_rule(rule_id, checked, violations, maximum, action):
    share = f"{violations / checked:.3%}" if checked else "N/A"
    comparison = ("Превышен" if violations > maximum else "Не превышен") if checked else "N/A"
    row = (rule_id, checked, violations, share, f"count ≤ {maximum}", comparison, action)
    rule_results[rule_id] = row
    show_table(["Правило", "Проверено", "Нарушений", "Доля", "Порог", "Сравнение", "Действие"],
               [row])

### Правило 1. `date_and_hour_present`

- **Поля:** `Date`, `Hour`.
- **Проверяемые строки:** все часовые записи.
- **Условие:** обе компоненты бизнес-ключа `(Date, Hour)` заполнены.
- **Порог:** число нарушений; переменная `max_missing_keys = 0`.
- **Критичность:** `critical`.
- **Действие:** `quarantine`.
- **Связь с задачей:** без даты и часа невозможно построить сезонный срез (`Seasons`) и суточный профиль (`Hour`), а также проверить уникальность бизнес-ключа; такие строки искажают средние по срезам Project Brief.

In [5]:
max_missing_keys = 0
checked, violations = connection.execute("""
    SELECT count(*), count(*) FILTER (
        WHERE "Date" IS NULL OR "Hour" IS NULL
    ) FROM raw_data
""").fetchone()
show_rule("date_and_hour_present", checked, violations, max_missing_keys, "quarantine")

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Действие
date_and_hour_present,8760,0,0.000%,count ≤ 0,Не превышен,quarantine


### Правило 2. `valid_hour_range`

- **Поле:** `Hour`.
- **Проверяемые строки:** часовые записи с заполненным `Hour`.
- **Условие:** `0 ≤ Hour ≤ 23`.
- **Порог:** число нарушений; переменная `max_bad_hour = 0`.
- **Критичность:** `critical`.
- **Действие:** `quarantine`.
- **Связь с задачей:** метрика «Среднее число аренд по часу» и суточный профиль из Project Brief строятся только по значениям из диапазона 0–23; час вне диапазона не попадает ни в один срез.

In [6]:
max_bad_hour = 0
checked, violations = connection.execute("""
    SELECT count(*), count(*) FILTER (
        WHERE "Hour" < 0 OR "Hour" > 23
    ) FROM raw_data
    WHERE "Hour" IS NOT NULL
""").fetchone()
show_rule("valid_hour_range", checked, violations, max_bad_hour, "quarantine")

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Действие
valid_hour_range,8760,0,0.000%,count ≤ 0,Не превышен,quarantine


### Правило 3. `valid_seasons_category`

- **Поле:** `Seasons`.
- **Проверяемые строки:** часовые записи с заполненным `Seasons`.
- **Условие:** `Seasons IN ('Winter', 'Spring', 'Summer', 'Autumn')`.
- **Порог:** число нарушений; переменная `max_bad_season = 0`.
- **Критичность:** `critical`.
- **Действие:** `quarantine`.
- **Связь с задачей:** метрика «Доля аренд по сезону» и уточняющий вопрос 1 из Project Brief опираются на фиксированный справочник из четырёх сезонов; значение вне справочника потеряется при группировке и исказит сезонные средние.

In [7]:
max_bad_season = 0
checked, violations = connection.execute("""
    SELECT count(*), count(*) FILTER (
        WHERE "Seasons" NOT IN ('Winter', 'Spring', 'Summer', 'Autumn')
    ) FROM raw_data
    WHERE "Seasons" IS NOT NULL
""").fetchone()
show_rule("valid_seasons_category", checked, violations, max_bad_season, "quarantine")

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Действие
valid_seasons_category,8760,0,0.000%,count ≤ 0,Не превышен,quarantine


### Правило 4. `non_negative_rented_bike_count`

- **Поле:** `Rented Bike Count`.
- **Проверяемые строки:** часовые записи с заполненным числом аренд.
- **Условие:** `Rented Bike Count ≥ 0`.
- **Порог:** число нарушений; переменная `max_negative_rents = 0`.
- **Критичность:** `critical`.
- **Действие:** `quarantine`.
- **Связь с задачей:** основная метрика спроса входит в средние, суммы и доли из Project Brief; отрицательное значение физически невозможно и искажает все сезонные и суточные агрегаты.

In [8]:
max_negative_rents = 0
checked, violations = connection.execute("""
    SELECT count(*), count(*) FILTER (WHERE "Rented Bike Count" < 0)
    FROM raw_data WHERE "Rented Bike Count" IS NOT NULL
""").fetchone()
show_rule("non_negative_rented_bike_count", checked, violations, max_negative_rents, "quarantine")

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Действие
non_negative_rented_bike_count,8760,0,0.000%,count ≤ 0,Не превышен,quarantine


### Правило 5. `functioning_day_consistency`

- **Поля:** `Functioning Day`, `Rented Bike Count`.
- **Проверяемые строки:** часовые записи с заполненными `Functioning Day` и `Rented Bike Count`.
- **Условие:** если `Functioning Day = false`, то `Rented Bike Count = 0`.
- **Порог:** число нарушений; переменная `max_inconsistent_days = 0`.
- **Критичность:** `warning`.
- **Действие:** `flag`.
- **Связь с задачей:** в нерабочие часы ноль отражает недоступность сервиса, а не отсутствие спроса. Строку с ненулевой арендой в нерабочий час нужно пометить, чтобы не смешивать её со спросом при расчёте среднего.

Флаг означает нарушение условия, а не подтверждённую ошибку.
Запись остаётся с флагом, если другие правила не требуют карантина.

In [9]:
max_inconsistent_days = 0
checked, violations = connection.execute("""
    SELECT count(*), count(*) FILTER (
        WHERE NOT "Functioning Day" AND "Rented Bike Count" > 0
    ) FROM raw_data
    WHERE "Functioning Day" IS NOT NULL AND "Rented Bike Count" IS NOT NULL
""").fetchone()
show_rule("functioning_day_consistency", checked, violations, max_inconsistent_days, "flag")

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Действие
functioning_day_consistency,8760,0,0.000%,count ≤ 0,Не превышен,flag


### Правило 6. `no_duplicate_business_key`

- **Поля:** `Date`, `Hour` — составной бизнес-ключ.
- **Проверяемые строки:** весь raw-набор.
- **Условие:** каждая пара `(Date, Hour)` встречается ровно один раз.
- **Порог:** число избыточных строк по ключу; переменная `max_key_duplicates = 0`.
- **Критичность:** `critical`.
- **Действие:** `quarantine`.
- **Связь с задачей:** гранулярность — один час системы в сутки; дубликат пары `(Date, Hour)` приведёт к двойному учёту спроса и завысит суммарное и среднее число аренд в сезонных и суточных срезах.

In [10]:
max_key_duplicates = 0
checked, violations = connection.execute("""
    SELECT (SELECT count(*) FROM raw_data),
           COALESCE(sum(cnt - 1), 0) FROM (
        SELECT count(*) AS cnt FROM raw_data GROUP BY "Date", "Hour" HAVING count(*) > 1
    )
""").fetchone()
show_rule("no_duplicate_business_key", checked, violations, max_key_duplicates, "quarantine")
show_table(["Правило", "Проверено", "Нарушений", "Доля", "Порог", "Сравнение", "Действие"],
           rule_results.values())

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Действие
no_duplicate_business_key,8760,0,0.000%,count ≤ 0,Не превышен,quarantine


Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Действие
date_and_hour_present,8760,0,0.000%,count ≤ 0,Не превышен,quarantine
valid_hour_range,8760,0,0.000%,count ≤ 0,Не превышен,quarantine
valid_seasons_category,8760,0,0.000%,count ≤ 0,Не превышен,quarantine
non_negative_rented_bike_count,8760,0,0.000%,count ≤ 0,Не превышен,quarantine
functioning_day_consistency,8760,0,0.000%,count ≤ 0,Не превышен,flag
no_duplicate_business_key,8760,0,0.000%,count ≤ 0,Не превышен,quarantine


<a id="prepare"></a>
## Шаг 4. Нормализованные данные и карантин

В этом снимке аналитические имена и типы уже подходят для дальнейшего анализа:
`Date` — `DATE`, `Hour` и `Rented Bike Count` — целочисленные, `Seasons` — `VARCHAR`,
`Functioning Day` — `BOOLEAN`. Приведения типов и переименований не требуется.

Два результата строятся из `checked_data`:

- **staging** — строки без критических нарушений.
  Действие `quarantine` применяется к правилам `date_and_hour_present`, `valid_hour_range`,
  `valid_seasons_category`, `non_negative_rented_bike_count`, `no_duplicate_business_key`.
  Действие `flag` применяется к `functioning_day_consistency` через Boolean-признак
  `quality_functioning_day_consistency_violated`.
- **quarantine** — строки, нарушившие хотя бы одно критическое правило.
  Каждая запись сохраняется один раз. Все причины перечисляются через `;` в полях
  `quality_rule_ids` (идентификаторы правил) и `quarantine_reason` (читаемое объяснение).
  Предупреждающее правило `functioning_day_consistency` в карантин не попадает.

Приоритет: если запись нарушает и критическое правило, и правило с действием `flag`,
она исключается в карантин, а признак `quality_functioning_day_consistency_violated`
для неё в staging не выставляется. Бизнес-агрегаций и «исправлений по догадке» нет.

Файл карантина создаётся при каждом запуске, даже если исключённых строк нет:
DuckDB сохраняет схему при `COPY` с нулевым числом строк. В `staging` дополнительно
присутствует техническое поле `ingested_on`, которое DuckDB получил из имени каталога.

`COALESCE(..., false)` в `checked_data` относится к результату сравнения, а не заменяет исходное значение нулём.

In [11]:
connection.execute("""
    CREATE OR REPLACE TEMP VIEW checked_data AS
    SELECT *,
        "Date" IS NULL OR "Hour" IS NULL AS missing_keys,
        COALESCE("Hour" < 0 OR "Hour" > 23, false) AS bad_hour,
        COALESCE("Seasons" NOT IN ('Winter', 'Spring', 'Summer', 'Autumn'), false) AS bad_season,
        COALESCE("Rented Bike Count" < 0, false) AS negative_rents,
        COALESCE(NOT "Functioning Day" AND "Rented Bike Count" > 0, false) AS inconsistent_day,
        COUNT(*) OVER (PARTITION BY "Date", "Hour") > 1 AS key_duplicate
    FROM raw_data
""")
connection.execute("""
    CREATE OR REPLACE TEMP VIEW staging_data AS
    SELECT * EXCLUDE (missing_keys, bad_hour, bad_season, negative_rents, inconsistent_day, key_duplicate),
        inconsistent_day AS quality_functioning_day_consistency_violated
    FROM checked_data
    WHERE NOT (missing_keys OR bad_hour OR bad_season OR negative_rents OR key_duplicate)
""")
connection.execute("""
    CREATE OR REPLACE TEMP VIEW quarantine_data AS
    SELECT * EXCLUDE (missing_keys, bad_hour, bad_season, negative_rents, inconsistent_day, key_duplicate),
        concat(
            CASE WHEN missing_keys THEN 'date_and_hour_present;' ELSE '' END,
            CASE WHEN bad_hour THEN 'valid_hour_range;' ELSE '' END,
            CASE WHEN bad_season THEN 'valid_seasons_category;' ELSE '' END,
            CASE WHEN negative_rents THEN 'non_negative_rented_bike_count;' ELSE '' END,
            CASE WHEN key_duplicate THEN 'no_duplicate_business_key;' ELSE '' END
        ) AS quality_rule_ids,
        concat(
            CASE WHEN missing_keys THEN 'Не заполнены дата или час; ' ELSE '' END,
            CASE WHEN bad_hour THEN 'Час вне диапазона 0-23; ' ELSE '' END,
            CASE WHEN bad_season THEN 'Сезон вне справочника; ' ELSE '' END,
            CASE WHEN negative_rents THEN 'Отрицательное число аренд; ' ELSE '' END,
            CASE WHEN key_duplicate THEN 'Дубликат бизнес-ключа (Date, Hour); ' ELSE '' END
        ) AS quarantine_reason
    FROM checked_data
    WHERE missing_keys OR bad_hour OR bad_season OR negative_rents OR key_duplicate
""")
show_table(["Набор", "Строк"], [
    (name, connection.execute(f"SELECT count(*) FROM {name}_data").fetchone()[0])
    for name in ("raw", "staging", "quarantine")
])

Набор,Строк
raw,8760
staging,8760
quarantine,0


In [12]:
show_table(["Date", "Hour", "Seasons", "Rented Bike Count", "Флаг"], connection.execute("""
    SELECT "Date", "Hour", "Seasons", "Rented Bike Count",
           quality_functioning_day_consistency_violated
    FROM staging_data
    ORDER BY "Date", "Hour" LIMIT 2
""").fetchall())
show_table(["Карантин", "Строк"], [
    ("пусто", connection.execute("SELECT count(*) FROM quarantine_data").fetchone()[0])
])

Date,Hour,Seasons,Rented Bike Count,Флаг
2017-12-01,0,Winter,254,False
2017-12-25,12,Winter,0,False


Карантин,Строк
пусто,0


### Запись двух результатов в S3

Пути вывода:

- `s3://staging/seoul_bike/source_ingested_on=2026-10-15/staging.parquet`
- `s3://staging/seoul_bike/source_ingested_on=2026-10-15/quarantine.parquet`

Логическая дата совпадает с `ingested_on` исходного raw-объекта (а не с датой запуска):
префикс вычислен из `raw_path`, что защищает от расхождения. Следующая ячейка
удаляет оба объекта и создаёт их заново — повторный прогон не добавляет дубликатов.

In [13]:
expected_prefix = f"s3://staging/seoul_bike/source_ingested_on={source_date}"
if output_prefix != expected_prefix:
    raise ValueError("Выходной префикс не соответствует исходному снимку.")

endpoint = os.environ["S3_ENDPOINT"]
client = boto3.client(
    "s3", endpoint_url=endpoint if endpoint.startswith("http") else f"http://{endpoint}",
    aws_access_key_id=os.environ["S3_ACCESS_KEY"],
    aws_secret_access_key=os.environ["S3_SECRET_KEY"], region_name="us-east-1",
)
prefix = output_prefix.removeprefix("s3://staging/")
for name in ("staging", "quarantine"):
    key = f"{prefix}/{name}.parquet"
    client.delete_object(Bucket="staging", Key=key)
    connection.execute(
        f"COPY (SELECT * FROM {name}_data) TO {literal('s3://staging/' + key)} (FORMAT PARQUET)"
    )
show_table(["Результат", "S3-путь"], [
    (name, f"{output_prefix}/{name}.parquet") for name in ("staging", "quarantine")
])

Результат,S3-путь
staging,s3://staging/seoul_bike/source_ingested_on=2026-10-15/staging.parquet
quarantine,s3://staging/seoul_bike/source_ingested_on=2026-10-15/quarantine.parquet


<a id="report"></a>
## Шаг 5. Краткий отчёт

Перенесите таблицы и значимые наблюдения в [docs/data_quality_report.md](../docs/data_quality_report.md).
Отчёт содержит пять разделов: контекст, профиль, правила и результаты, подготовленные данные,
воспроизводимость и ограничения. Полные описания правил и SQL повторять не нужно — сошлитесь на этот ноутбук.

В raw все шесть проверок пройдены: пропусков ключа, значения часа вне диапазона, сезонов вне справочника,
отрицательных аренд, несогласованных нерабочих часов и дубликатов бизнес-ключа нет.
Карантин пуст, предупреждений нет.

<a id="verify"></a>
## Шаг 6. Проверка сохранённых файлов

Читаем оба файла непосредственно из S3.
Проверяем баланс, отсутствие критических нарушений, правильность флага и причины карантина.
Количество полей при чтении может включать `source_ingested_on` из выходного пути —
его нет внутри Parquet-файлов.

In [14]:
for name in ("staging", "quarantine"):
    path = f"{output_prefix}/{name}.parquet"
    connection.execute(
        f"CREATE OR REPLACE TEMP VIEW saved_{name} AS SELECT * FROM read_parquet({literal(path)})"
    )
staging_count = connection.execute("SELECT count(*) FROM saved_staging").fetchone()[0]
quarantine_count = connection.execute("SELECT count(*) FROM saved_quarantine").fetchone()[0]
critical_count = connection.execute("""
    SELECT count(*) FROM saved_staging
    WHERE "Date" IS NULL OR "Hour" IS NULL
       OR "Hour" < 0 OR "Hour" > 23
       OR "Seasons" NOT IN ('Winter', 'Spring', 'Summer', 'Autumn')
       OR "Rented Bike Count" < 0
""").fetchone()[0]
staging_key_dups = connection.execute("""
    SELECT COALESCE(sum(cnt - 1), 0) FROM (
        SELECT count(*) AS cnt FROM saved_staging GROUP BY "Date", "Hour" HAVING count(*) > 1
    )
""").fetchone()[0]
bad_flags = connection.execute("""
    SELECT count(*) FROM saved_staging
    WHERE quality_functioning_day_consistency_violated
          IS DISTINCT FROM COALESCE(NOT "Functioning Day" AND "Rented Bike Count" > 0, false)
""").fetchone()[0]
flagged_count = connection.execute("""
    SELECT count(*) FROM saved_staging
    WHERE quality_functioning_day_consistency_violated
""").fetchone()[0]
bad_reasons = connection.execute("""
    SELECT count(*) FROM saved_quarantine
    WHERE quality_rule_ids IS NULL OR quality_rule_ids = ''
       OR quarantine_reason IS NULL OR quarantine_reason = ''
""").fetchone()[0]

assert raw_count == staging_count + quarantine_count, "Не сошёлся баланс строк."
assert critical_count == 0, "В staging остались критические нарушения."
assert staging_key_dups == 0, "В staging остались дубликаты бизнес-ключа."
assert bad_flags == 0, "Некорректные признаки нерабочих дней."
assert bad_reasons == 0, "В карантине есть строки без причины."

run_summary = (raw_count, staging_count, quarantine_count, flagged_count, critical_count)
show_table(["Raw", "Staging", "Карантин", "Предупреждения", "Критические нарушения"], [run_summary])
show_table(["Проверка", "Нарушений"], [
    ("Неверные флаги", bad_flags), ("Пустые причины карантина", bad_reasons)
])

Raw,Staging,Карантин,Предупреждения,Критические нарушения
8760,8760,0,0,0


Проверка,Нарушений
Неверные флаги,0
Пустые причины карантина,0


In [15]:
show_table(["Date", "Hour", "Rented Bike Count", "Флаг"], connection.execute("""
    SELECT "Date", "Hour", "Rented Bike Count",
           quality_functioning_day_consistency_violated
    FROM saved_staging
    ORDER BY "Date", "Hour" LIMIT 2
""").fetchall())
show_table(["Карантин", "Строк"], [
    ("пусто", connection.execute("SELECT count(*) FROM saved_quarantine").fetchone()[0])
])

Date,Hour,Rented Bike Count,Флаг
2017-12-01,0,254,False
2017-12-25,12,0,False


Карантин,Строк
пусто,0


### Повторное выполнение с чистого ядра

Запишите итоговые количества первого выполнения в отчёт. Перезапустите ядро и выполните **все ячейки сверху вниз**
с теми же входным путём, периодом, порогами и выходным префиксом.
Сравните итоговую таблицу и добавьте второй результат в отчёт.

Контрольные количества для этого снимка: raw — 8 760, staging — 8 760, карантин — 0,
предупреждения в staging — 0, критические нарушения — 0.
После проверки сохраните ноутбук с выводами таблиц.